# Exercise 21: Immediate Food Delivery II

**Source:** [LeetCode 1174](https://leetcode.com/problems/immediate-food-delivery-ii/description/?envType=study-plan-v2&envId=top-sql-50) (Medium)

Table: `Delivery`
```
+-----------------------------+---------+
| Column Name                 | Type    |
+-----------------------------+---------+
| delivery_id                 | int     |
| customer_id                 | int     |
| order_date                  | date    |
| customer_pref_delivery_date | date    |
+-----------------------------+---------+
```
- `delivery_id` is the primary key.
- `order_date` is the date when the customer `customer_id` ordered food.
- `customer_pref_delivery_date` is the preferred delivery date.

An order is called **immediate** if `order_date` = `customer_pref_delivery_date`.

A **first order** is the earliest date of an order by a fixed `customer_id`.

**Task.**
Find the percentage of immediate orders among all first orders rounded to two decimals.

Connect to the database `postgres` using the username `postgres`.

In [1]:
%load_ext sql
%sql postgresql+psycopg://postgres@localhost:5432/postgres

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'postgresql+psycopg://postgres@localhost:5432/postgres'

### Create new database for the exercise

Make sure first that a database with the same name is removed in case it already exists.

In [2]:
%sql DROP DATABASE IF EXISTS exercise_21 WITH (FORCE);
%sql CREATE DATABASE exercise_21;

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

Connect to the database.

*Caution.* No semicolon at the end otherwise it belongs to the name of the database.

In [3]:
%sql postgresql+psycopg://postgres@localhost:5432/exercise_21 --alias exercise_21

Connecting and switching to connection 'exercise_21'

#### Fill the database with content

In [4]:
%%sql
CREATE TABLE IF NOT EXISTS Delivery (
    delivery_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date DATE NOT NULL,
    customer_pref_delivery_date DATE
);

INSERT INTO Delivery (delivery_id, customer_id, order_date, customer_pref_delivery_date)
VALUES
(1, 1, '2019-08-01', '2019-08-02'),
(2, 2, '2019-08-02', '2019-08-02'),
(3, 1, '2019-08-11', '2019-08-12'),
(4, 3, '2019-08-24', '2019-08-24'),
(5, 3, '2019-08-21', '2019-08-22'),
(6, 2, '2019-08-11', '2019-08-13'),
(7, 4, '2019-08-09', '2019-08-09');

Running query in 'exercise_21'

7 rows affected.

++
||
++
++

#### Solve the exercise

**Task.**
Find the percentage of immediate orders among all first orders rounded to two decimals.

In [5]:
%%sql
SELECT
    ROUND(COUNT(CASE WHEN order_date = customer_pref_delivery_date THEN 1 END)::NUMERIC/COUNT(*) * 100, 2) AS immediate_percentage
FROM (
    SELECT
        customer_id, order_date, customer_pref_delivery_date
    FROM
        Delivery D
    WHERE order_date = (SELECT MIN(C.order_date) FROM Delivery C WHERE C.customer_id = D.customer_id)
    );

Running query in 'exercise_21'

1 rows affected.

immediate_percentage
50.00


#### Explanation

- First select for each `customer_id` the first order. This id done via subquery selecting `MIN(order_date)` for every `customer_id` using a `WHERE`-clause:

In [6]:
%%sql
SELECT
    customer_id, order_date, customer_pref_delivery_date
FROM
    Delivery D
WHERE order_date = (SELECT MIN(C.order_date) FROM Delivery C WHERE C.customer_id = D.customer_id)

Running query in 'exercise_21'

4 rows affected.

customer_id,order_date,customer_pref_delivery_date
1,2019-08-01,2019-08-02
2,2019-08-02,2019-08-02
3,2019-08-21,2019-08-22
4,2019-08-09,2019-08-09


- Counting over the rows with `order_date = customer_pref_delivery_date` (do this using `CASE WHEN ... THEN 1 END`),
and dividing over the total count of the above table, one obtains the fraction.
Then multiply with `100` and apply `ROUND(_,2)`

#### Close connection

In [7]:
%sql --close exercise_21
%sql postgresql+psycopg://postgres@localhost:5432/postgres
%sql DROP DATABASE IF EXISTS exercise_21;

Switching to connection 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

**Well done!** (approved by Leet)